# NB01 — Data Acquisition

**Phase 1.** Download and filter the three public-data sources that supply the agent corpus.

Run cells top-to-bottom. Each section prints a summary to inspect before proceeding.

| Source | Licence | Use |
|---|---|---|
| Yelp Open Dataset | Academic (manual download required) | 500 Italian restaurants + reviews, Philadelphia + Tampa |
| NYPL "What's on the Menu?" | CC0 | ~200 historical menus for art-director reference |
| USDA FoodData Central SR Legacy | Public domain | Ingredient grounding for copywriter agent |

## 1. Environment setup

Resolve project root, load `.env`, verify the venv has all required packages.

In [ ]:
from pathlib import Path
import json
import os
import sys
import zipfile

import requests
import pandas as pd
from tqdm.auto import tqdm
from dotenv import load_dotenv

# Resolve project root (works whether notebook is run from menuforge/ or notebooks/)
ROOT = Path().resolve()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent
assert (ROOT / "configs").exists(), (
    f"Cannot find project root from {Path().resolve()}. "
    "Run from menuforge/ or menuforge/notebooks/."
)

load_dotenv(ROOT / ".env", override=False)
print(f"Project root : {ROOT}")
print(f"Python       : {sys.version.split()[0]}")
print(f"pandas       : {pd.__version__}")

## 2. Configuration

Central place for all paths and acquisition parameters. Edit here if you want to change metros, cuisine, or sample sizes — nothing below this cell contains hard-coded values.

In [ ]:
# ── Directories ────────────────────────────────────────────────────────────────
RAW       = ROOT / "data" / "raw"
PROCESSED = ROOT / "data" / "processed"
YELP_DIR  = RAW / "yelp"
NYPL_DIR  = RAW / "nypl"
USDA_DIR  = RAW / "usda"
IMAGES    = RAW / "menu_images"

for d in [RAW, PROCESSED, YELP_DIR, NYPL_DIR, USDA_DIR, IMAGES]:
    d.mkdir(parents=True, exist_ok=True)

# ── Yelp source files ──────────────────────────────────────────────────────────
YELP_BUSINESS = YELP_DIR / "yelp_academic_dataset_business.json"
YELP_REVIEWS  = YELP_DIR / "yelp_academic_dataset_review.json"

# ── Target metros: city → expected state ──────────────────────────────────────
METROS = {
    "Philadelphia": "PA",
    "Tampa":        "FL",
}
CUISINE = "Italian"
PHOTO_N = 30  # top-N candidates by review_count for image download

# ── External dataset URLs ──────────────────────────────────────────────────────
NYPL_BASE = "https://s3.amazonaws.com/menusdata.nypl.org/gzipped"
NYPL_FILES = ["Menu.csv.gz", "MenuItem.csv.gz", "MenuPage.csv.gz", "Dish.csv.gz"]
USDA_URL   = (
    "https://fdc.nal.usda.gov/fdc-datasets/"
    "FoodData_Central_SR_Legacy_csv_2019-04-02.zip"
)

print("Directories ready:")
for d in [RAW, PROCESSED, YELP_DIR, NYPL_DIR, USDA_DIR, IMAGES]:
    print(f"  {d.relative_to(ROOT)}")

## 3. Yelp — check raw files

Confirms the Yelp business and review JSONL files are present and shows their sizes. **This cell will raise `FileNotFoundError` with download instructions if the files are missing** — the Yelp dataset requires a manual academic-licence request.

In [ ]:
missing = []
for f in [YELP_BUSINESS, YELP_REVIEWS]:
    if f.exists():
        size_gb = f.stat().st_size / 1e9
        print(f"  OK      {f.name}  ({size_gb:.2f} GB)")
    else:
        print(f"  MISSING {f.name}")
        missing.append(f)

if missing:
    print()
    print("Yelp dataset files not found. To download:")
    print("  1. Go to https://www.yelp.com/dataset")
    print("  2. Request the dataset (free, academic licence).")
    print("  3. Extract the JSON files and place them in:")
    print(f"     {YELP_DIR}")
    print("  4. Re-run this cell.")
    raise FileNotFoundError(f"Missing Yelp files: {[f.name for f in missing]}")

print("\nAll Yelp source files present.")

## 4. Yelp — filter businesses to target metros and cuisine

Streams the business JSON line-by-line and keeps only restaurants where `city` is Philadelphia (PA) or Tampa (FL) and `"Italian"` appears in the `categories` field. Extracts the `RestaurantsPriceRange2` attribute as the price tier signal.

In [ ]:
records = []
with open(YELP_BUSINESS, encoding="utf-8") as f:
    for line in tqdm(f, desc="Scanning businesses", unit=" biz"):
        biz   = json.loads(line)
        city  = biz.get("city", "")
        state = biz.get("state", "")
        cats  = biz.get("categories") or ""
        if city in METROS and METROS[city] == state and CUISINE in cats:
            attrs = biz.get("attributes") or {}
            records.append({
                "business_id":  biz["business_id"],
                "name":         biz["name"],
                "address":      biz.get("address", ""),
                "city":         city,
                "state":        state,
                "postal_code":  biz.get("postal_code", ""),
                "latitude":     biz.get("latitude"),
                "longitude":    biz.get("longitude"),
                "stars":        biz.get("stars"),
                "review_count": biz.get("review_count"),
                "is_open":      biz.get("is_open"),
                "categories":   cats,
                "price_range":  attrs.get("RestaurantsPriceRange2"),
            })

restaurants = pd.DataFrame(records)
print(f"Total Italian restaurants found: {len(restaurants)}")
print()
print(restaurants["city"].value_counts().to_string())

## 5. Inspect restaurant distribution

Check star ratings, review counts, price tier spread, and open/closed ratio per metro. **Gate check: ≥200 restaurants per metro.** If a metro is below target, check whether the city name has variants in the data (e.g. surrounding suburbs) — a note is printed if so.

In [ ]:
for city, grp in restaurants.groupby("city"):
    print(f"── {city} ({len(grp)} restaurants) ──")
    s = grp["stars"]
    print(f"  Stars        mean={s.mean():.2f}  min={s.min()}  max={s.max()}")
    rc = grp["review_count"]
    print(f"  Review count median={rc.median():.0f}  max={rc.max()}")
    pr = grp["price_range"].value_counts(dropna=False).sort_index()
    print(f"  Price range  {pr.to_dict()}")
    io = grp["is_open"].value_counts().to_dict()
    print(f"  is_open      {io}")
    print()

# ── Gate check ────────────────────────────────────────────────────────────────
print("── Gate: ≥200 restaurants per metro ──")
for city in METROS:
    n = (restaurants["city"] == city).sum()
    status = "OK" if n >= 200 else "BELOW TARGET"
    print(f"  {city}: {n}  [{status}]")

# ── Check for city-name variants that may have been excluded ──────────────────
print()
print("Note — if a metro is below target, check these nearby city names in the raw data:")
nearby = {
    "Philadelphia": ["South Philadelphia", "West Philadelphia", "North Philadelphia", "Phila"],
    "Tampa":        ["Ybor City", "Hyde Park", "Westshore"],
}
for city, variants in nearby.items():
    print(f"  {city}: {variants}  (would need to broaden METROS filter if needed)")

## 6. Save restaurants.parquet

Write the filtered restaurant dataframe to `data/processed/restaurants.parquet`. Prints schema and file size to confirm write succeeded.

In [ ]:
out = PROCESSED / "restaurants.parquet"
restaurants.to_parquet(out, index=False)
size_kb = out.stat().st_size / 1024
print(f"Saved : {out.relative_to(ROOT)}")
print(f"Size  : {size_kb:.1f} KB")
print(f"Shape : {restaurants.shape}")
print()
print(restaurants.dtypes.to_string())

## 7. Stream and filter reviews

The Yelp review file is ~6 GB. This cell streams it line-by-line and keeps only reviews whose `business_id` is in our filtered restaurant set. Memory usage stays flat regardless of file size.

In [ ]:
target_ids = set(restaurants["business_id"])
print(f"Filtering reviews for {len(target_ids)} restaurants...")

review_records = []
with open(YELP_REVIEWS, encoding="utf-8") as f:
    for line in tqdm(f, desc="Streaming reviews", unit=" reviews"):
        rv = json.loads(line)
        if rv["business_id"] in target_ids:
            review_records.append({
                "review_id":   rv["review_id"],
                "business_id": rv["business_id"],
                "user_id":     rv["user_id"],
                "stars":       rv["stars"],
                "text":        rv["text"],
                "date":        rv["date"],
                "useful":      rv.get("useful", 0),
                "funny":       rv.get("funny", 0),
                "cool":        rv.get("cool", 0),
            })

reviews = pd.DataFrame(review_records)
print(f"\nTotal reviews matched: {len(reviews)}")
print()
print("Star distribution:")
print(reviews["stars"].value_counts().sort_index().to_string())

## 8. Inspect review distribution

Reviews per metro, median text length, and a sample review for a sanity check on text quality. These numbers inform Phase 2 EDA.

In [ ]:
# Join to get city labels
city_map = restaurants.set_index("business_id")["city"]
reviews["city"] = reviews["business_id"].map(city_map)

print("Reviews per city:")
print(reviews["city"].value_counts().to_string())
print()

text_len = reviews["text"].str.len()
print(f"Text length  median={text_len.median():.0f}  mean={text_len.mean():.0f}  max={text_len.max()}")
print()

# Date range
reviews["date"] = pd.to_datetime(reviews["date"])
print(f"Date range   {reviews['date'].min().date()} → {reviews['date'].max().date()}")
print()

# Sample
sample = reviews[reviews["stars"] == 3]["text"].iloc[0]
print("Sample 3-star review (first 300 chars):")
print(sample[:300])

## 9. Save reviews.parquet

Write filtered reviews to `data/processed/reviews.parquet`. The `city` column added in the previous cell is included — it avoids repeated joins in later notebooks.

In [ ]:
out = PROCESSED / "reviews.parquet"
reviews.to_parquet(out, index=False)
size_mb = out.stat().st_size / 1e6
print(f"Saved : {out.relative_to(ROOT)}")
print(f"Size  : {size_mb:.1f} MB")
print(f"Shape : {reviews.shape}")
print()
print(reviews.dtypes.to_string())

## 10. Yelp menu photos — extract from dataset

Filters `photos.json` to `label == 'menu'` only, builds an extraction manifest,
cleans up any non-menu photos from earlier runs, and extracts the matching files
from the tar.

**Finding for the scan:** only 16 of 200,000 Yelp photos in our segment are
menu-labelled (0.008%). Public photo corpora are insufficient for menu-specific
VLM tasks without a dedicated menu image source — which is what NYPL provides.

In [ ]:
import tarfile
from collections import Counter

PHOTOS_TAR  = IMAGES / "yelp_photos.tar"
PHOTOS_JSON = IMAGES / "photos.json"

if not PHOTOS_JSON.exists():
    print("Extracting photos.json from tar (one-time)...")
    with tarfile.open(PHOTOS_TAR, "r:gz") as tf:
        data = tf.extractfile(tf.getmember("photos.json")).read()
        PHOTOS_JSON.write_bytes(data)

# Build photo index — menu-labelled only
target_ids = set(restaurants["business_id"])
biz_photos = {}
with open(PHOTOS_JSON, encoding="utf-8") as f:
    for line in f:
        p = json.loads(line)
        if p["business_id"] in target_ids and p["label"] == "menu":
            biz_photos.setdefault(p["business_id"], []).append(p)

print(f"Restaurants in filtered set : {len(target_ids)}")
print(f"Restaurants with menu photos: {len(biz_photos)}")
print(f"Total menu-labelled photos  : {sum(len(v) for v in biz_photos.values())}")
print()
print("Note: only 16 of 200k Yelp photos are menu-labelled in our segment.")
print("This is documented as a finding. Chain restaurant supplement follows.")

# Build extraction manifest
to_extract = {}
for bid, photos in biz_photos.items():
    for p in photos:
        to_extract[f"photos/{p['photo_id']}.jpg"] = (bid, p["photo_id"])

print(f"\nPhotos to extract: {len(to_extract)}")

In [ ]:
# Step 1 — clean up any non-menu photos from previous extraction
import shutil

PHOTOS_JSON = IMAGES / "photos.json"
extracted_ids = {p.stem for p in IMAGES.glob("**/*.jpg")}

menu_ids = set()
with open(PHOTOS_JSON, encoding="utf-8") as f:
    for line in f:
        p = json.loads(line)
        if p["photo_id"] in extracted_ids and p["label"] == "menu":
            menu_ids.add(p["photo_id"])

removed = 0
for jpg in list(IMAGES.glob("**/*.jpg")):
    if jpg.stem not in menu_ids:
        jpg.unlink()
        removed += 1

# Remove now-empty business folders
for d in [x for x in IMAGES.iterdir() if x.is_dir()]:
    if not any(d.iterdir()):
        d.rmdir()

print(f"Cleaned {removed} non-menu photos.")
print(f"Remaining Yelp menu photos: {len(list(IMAGES.glob('**/*.jpg')))}")

# Step 2 — extract any menu photos not yet present
already = {p.stem for p in IMAGES.glob("**/*.jpg")}
remaining = {k: v for k, v in to_extract.items() if v[1] not in already}

if not remaining:
    print("All menu photos already extracted.")
else:
    print(f"Extracting {len(remaining)} new menu photos from tar...")
    with tarfile.open(PHOTOS_TAR, "r:gz") as tf:
        members = {m.name: m for m in tqdm(tf.getmembers(), desc="Indexing", unit=" files")}
        for tar_path, (bid, photo_id) in tqdm(remaining.items(), desc="Extracting"):
            if tar_path not in members:
                continue
            out_dir = IMAGES / bid
            out_dir.mkdir(exist_ok=True)
            data = tf.extractfile(members[tar_path]).read()
            (out_dir / f"{photo_id}.jpg").write_bytes(data)

final = list(IMAGES.glob("**/*.jpg"))
print(f"\nYelp menu photos ready: {len(final)}")
for jpg in final:
    print(f"  {jpg.relative_to(IMAGES)}")

## 11. NYPL "What's on the Menu?" — download

Downloads the NYPL dataset (35 MB single `.tgz`) from the current S3 location
and extracts the four core CSV tables to `data/raw/nypl/`.

**Note:** The original `menus.nypl.org` site retired January 2025. The dataset
was archived to `nypl-menus-data.s3.amazonaws.com` — the URL hardcoded here is
the current stable location.

In [ ]:
NYPL_TGZ = NYPL_DIR / "nypl_menus_data.tgz"
NYPL_URL  = "https://nypl-menus-data.s3.amazonaws.com/gzips/2023_03_16_07_02_35_data.tgz"
NYPL_CORE = {"Menu", "MenuItem", "MenuPage", "Dish"}

# Download
if NYPL_TGZ.exists():
    print(f"Already downloaded: {NYPL_TGZ.name} ({NYPL_TGZ.stat().st_size / 1e6:.1f} MB)")
else:
    print(f"Downloading NYPL dataset (~35 MB)...")
    r = requests.get(NYPL_URL, stream=True, timeout=120)
    r.raise_for_status()
    with open(NYPL_TGZ, "wb") as f:
        for chunk in tqdm(r.iter_content(chunk_size=1 << 20), desc="NYPL", unit="MB"):
            f.write(chunk)
    print(f"Downloaded: {NYPL_TGZ.stat().st_size / 1e6:.1f} MB")

# Extract core tables
import tarfile
print("\nExtracting core tables...")
with tarfile.open(NYPL_TGZ, "r:gz") as tf:
    all_names = tf.getnames()
    print(f"Archive contains {len(all_names)} files. First 10: {all_names[:10]}")
    for member in tf.getmembers():
        stem = Path(member.name).stem
        if stem in NYPL_CORE:
            dest = NYPL_DIR / Path(member.name).name
            if dest.exists():
                print(f"  SKIP  {dest.name}")
                continue
            data = tf.extractfile(member)
            if data:
                dest.write_bytes(data.read())
                print(f"  OK    {dest.name}  ({dest.stat().st_size / 1e6:.1f} MB)")

# Show what's in NYPL dir
print("\nNYPL directory:")
for f in sorted(NYPL_DIR.iterdir()):
    if f.is_file() and f.suffix in (".csv", ".gz"):
        print(f"  {f.name}  ({f.stat().st_size / 1e6:.1f} MB)")

## 11b. NYPL — select and download menu page images

Uses the downloaded `MenuPage.csv.gz` to select ~30 menu page images and downloads
them from NYPL's image server. These become the VLM extraction corpus for Phase 3.

**Why NYPL for VLM evaluation:** `MenuItem.csv` already contains human-transcribed
dish names and prices — no manual labelling needed for the Phase 3 precision gate.
The model's extractions are compared directly against the NYPL ground truth.

**Cell A** selects pages and shows what will be downloaded before committing.
**Cell B** downloads the images (~30 requests, a few seconds each).

In [ ]:
NYPL_IMG_DIR = NYPL_DIR / "images"
NYPL_IMG_DIR.mkdir(exist_ok=True)
N_IMAGES = 30

# Find Menu and MenuPage files — accept .csv or .csv.gz
def find_csv(directory, stem):
    for suffix in (".csv", ".csv.gz"):
        p = directory / f"{stem}{suffix}"
        if p.exists():
            return p
    raise FileNotFoundError(f"{stem}.csv not found in {directory} — run cell 11 first.")

menu_file = find_csv(NYPL_DIR, "Menu")
page_file = find_csv(NYPL_DIR, "MenuPage")

compression = "gzip" if str(menu_file).endswith(".gz") else None
menus_df = pd.read_csv(menu_file, compression=compression, low_memory=False)
compression = "gzip" if str(page_file).endswith(".gz") else None
pages_df  = pd.read_csv(page_file,  compression=compression, low_memory=False)

print(f"Menu.csv     {len(menus_df):>6} rows | cols: {list(menus_df.columns)}")
print(f"MenuPage.csv {len(pages_df):>6} rows | cols: {list(pages_df.columns)}")

# Filter to Italian-related menus
text_cols = [c for c in menus_df.columns if menus_df[c].dtype == object]
italian_mask = pd.Series(False, index=menus_df.index)
for col in text_cols:
    italian_mask |= menus_df[col].fillna("").str.contains("[Ii]talian|[Pp]asta", regex=True)

italian_menus = menus_df[italian_mask]
print(f"\nItalian-related menus: {len(italian_menus)}")

source = italian_menus if len(italian_menus) >= N_IMAGES else menus_df
if len(italian_menus) < N_IMAGES:
    print(f"  Fewer than {N_IMAGES} — using full corpus sorted by dish_count")

dish_col = next((c for c in source.columns if "dish" in c.lower() and "count" in c.lower()), None)
if dish_col:
    source = source.sort_values(dish_col, ascending=False)

# One first page per menu
target_ids    = set(source["id"].head(N_IMAGES * 3))
candidate_pages = pages_df[pages_df["menu_id"].isin(target_ids)].copy()
page_num_col  = next((c for c in candidate_pages.columns if "page" in c.lower() and "num" in c.lower()), None)
if page_num_col:
    candidate_pages = (
        candidate_pages.sort_values(page_num_col)
        .groupby("menu_id", sort=False)
        .first()
        .reset_index()
    )

target_pages = candidate_pages.head(N_IMAGES).reset_index(drop=True)
print(f"\nPages selected: {len(target_pages)}")
print(target_pages.head(8).to_string())

In [ ]:
# Identify the image ID column
img_col = next((c for c in target_pages.columns if "image" in c.lower()), None)

if img_col is None:
    print("No image column found in MenuPage.csv — check the columns printed above.")
else:
    print(f"Using image column: '{img_col}'")
    downloaded, skipped, failed = 0, 0, []

    for _, row in tqdm(target_pages.iterrows(), total=len(target_pages), desc="Downloading"):
        page_id  = row["id"]
        image_id = row[img_col]
        dest     = NYPL_IMG_DIR / f"{page_id}.jpg"

        if dest.exists():
            skipped += 1
            continue

        url = f"https://images.nypl.org/index.php?id={image_id}&t=w"
        try:
            r = requests.get(url, timeout=30, headers={"User-Agent": "Mozilla/5.0"})
            r.raise_for_status()
            if "image" in r.headers.get("content-type", ""):
                dest.write_bytes(r.content)
                downloaded += 1
            else:
                failed.append(f"{page_id}: unexpected content-type {r.headers.get('content-type')}")
        except Exception as e:
            failed.append(f"{page_id}: {e}")

    imgs = list(NYPL_IMG_DIR.glob("*.jpg"))
    print(f"Downloaded : {downloaded}")
    print(f"Skipped    : {skipped} (already present)")
    print(f"Failed     : {len(failed)}")
    if failed:
        for msg in failed[:5]:
            print(f"  {msg}")
    print(f"\nNYPL menu images ready: {len(imgs)} → {NYPL_IMG_DIR.relative_to(ROOT)}")

## 12. USDA FoodData Central — download SR Legacy

Downloads the USDA Standard Reference Legacy dataset (stable 2019-04-02 release, ~40 MB) to `data/raw/usda/` and extracts the four core tables: `food`, `nutrient`, `food_nutrient`, `food_category`. These supply the ingredient grounding corpus for the copywriter agent in Phase 4.

In [ ]:
USDA_ZIP  = USDA_DIR / "FoodData_Central_sr_legacy.zip"
USDA_URL  = "https://fdc.nal.usda.gov/fdc-datasets/FoodData_Central_sr_legacy_food_csv_2018-04.zip"
USDA_CORE = {"food", "nutrient", "food_nutrient", "food_category"}

if USDA_ZIP.exists():
    print(f"Already present: {USDA_ZIP.name} ({USDA_ZIP.stat().st_size / 1e6:.1f} MB)")
else:
    print(f"Downloading USDA SR Legacy (~6 MB)...")
    r = requests.get(USDA_URL, stream=True, timeout=60)
    r.raise_for_status()
    with open(USDA_ZIP, "wb") as f:
        for chunk in tqdm(r.iter_content(chunk_size=1 << 20), desc="USDA", unit="MB"):
            f.write(chunk)
    print(f"Downloaded: {USDA_ZIP.stat().st_size / 1e6:.1f} MB")

# Extract core tables
import zipfile
print("\nExtracting core tables...")
with zipfile.ZipFile(USDA_ZIP) as z:
    all_names = z.namelist()
    print(f"Zip contains {len(all_names)} files. All: {all_names}")
    for name in all_names:
        stem = Path(name).stem.lower()
        if any(t in stem for t in USDA_CORE):
            dest = USDA_DIR / Path(name).name
            if dest.exists():
                print(f"  SKIP  {dest.name}")
                continue
            dest.write_bytes(z.read(name))
            print(f"  OK    {dest.name}  ({dest.stat().st_size / 1e6:.1f} MB)")

# Quick look at food table
food_files = sorted(USDA_DIR.glob("**/food.csv"))
if food_files:
    food_df = pd.read_csv(food_files[0], low_memory=False)
    print(f"\nUSDA food table: {len(food_df)} rows, cols: {list(food_df.columns)}")
    print(food_df.head(3).to_string())

## 13. Dataset card — acquisition summary

Reads back the saved parquet files and scans the raw directories to produce a concise dataset card. This is the evidence artefact for the Phase 1 gate check and for the horizon scan's data section.

In [ ]:
print("=" * 60)
print("DATASET CARD — Phase 1 Acquisition")
print("=" * 60)

# ── Yelp ─────────────────────────────────────────────────────────────────────
print("\n── Yelp Open Dataset ──")
rest_path = PROCESSED / "restaurants.parquet"
rev_path  = PROCESSED / "reviews.parquet"

if rest_path.exists():
    df_r = pd.read_parquet(rest_path)
    print(f"  restaurants.parquet : {len(df_r)} rows  ({rest_path.stat().st_size / 1e3:.0f} KB)")
    for city, n in df_r["city"].value_counts().items():
        print(f"    {city}: {n}")
else:
    print("  restaurants.parquet : NOT FOUND")

if rev_path.exists():
    df_v = pd.read_parquet(rev_path)
    print(f"  reviews.parquet     : {len(df_v)} rows  ({rev_path.stat().st_size / 1e6:.0f} MB)")
    for city, n in df_v["city"].value_counts().items():
        print(f"    {city}: {n}")
else:
    print("  reviews.parquet     : NOT FOUND")

# ── Yelp menu photos ──────────────────────────────────────────────────────────
print("\n── Menu images ──")
CHAIN_DIR = IMAGES / "chain_menus"
yelp_jpgs  = [p for p in IMAGES.rglob("*.jpg") if CHAIN_DIR not in p.parents]
chain_jpgs = list(CHAIN_DIR.glob("*.jpg")) if CHAIN_DIR.exists() else []
nypl_jpgs  = list((NYPL_DIR / "images").glob("*.jpg")) if (NYPL_DIR / "images").exists() else []
total_imgs = len(yelp_jpgs) + len(chain_jpgs) + len(nypl_jpgs)
print(f"  Yelp menu-labelled  : {len(yelp_jpgs)}  (from {len({p.parent for p in yelp_jpgs})} restaurants)")
print(f"  NYPL scanned menus  : {len(nypl_jpgs)}")
print(f"  Chain supplements   : {len(chain_jpgs)}")
print(f"  Total               : {total_imgs}")

# ── NYPL CSVs ────────────────────────────────────────────────────────────────
print("\n── NYPL What's on the Menu? ──")
nypl_csvs = sorted(NYPL_DIR.glob("*.csv")) + sorted(NYPL_DIR.glob("*.csv.gz"))
if nypl_csvs:
    for f in nypl_csvs:
        print(f"  {f.name}  ({f.stat().st_size / 1e6:.1f} MB)")
else:
    print("  No CSV files found — run cell 11.")

# ── USDA ─────────────────────────────────────────────────────────────────────
print("\n── USDA FoodData Central SR Legacy ──")
usda_core = ["food.csv", "nutrient.csv", "food_nutrient.csv", "food_category.csv"]
for name in usda_core:
    matches = list(USDA_DIR.glob(f"**/{name}"))
    if matches:
        f = matches[0]
        print(f"  {f.name}  ({f.stat().st_size / 1e6:.1f} MB)")
    else:
        print(f"  {name}  NOT FOUND")

# ── Gate checks ───────────────────────────────────────────────────────────────
print("\n── Phase 1 gate checks ──")
if rest_path.exists():
    for city in METROS:
        n = (df_r["city"] == city).sum()
        ok = n >= 200
        print(f"  ≥200 restaurants ({city}): {n}  [{'OK' if ok else 'FAIL'}]")
    pr_dist = df_r["price_range"].value_counts(dropna=True)
    print(f"  Non-degenerate price tiers: {pr_dist.to_dict()}  [{'OK' if len(pr_dist) >= 2 else 'FAIL'}]")
print(f"  ≥30 menu images total: {total_imgs}  [{'OK' if total_imgs >= 30 else f'PARTIAL — add NYPL images (cell 11b) or chain screenshots'}]")
nypl_ok = len(nypl_csvs) >= 4
print(f"  NYPL CSVs present (4): {len(nypl_csvs)}  [{'OK' if nypl_ok else 'FAIL — run cell 11'}]")
usda_ok = len(list(USDA_DIR.glob("**/food.csv"))) > 0
print(f"  USDA food.csv present: {'OK' if usda_ok else 'FAIL — run cell 12'}")
print()
print("Next: NB01b_lancedb_init.ipynb")